# MJX 01 — MuJoCo Concepts & MJCF Models

### Lab Description

This lab builds your mental model of the **MuJoCo** physics engine, the foundation for the whole MuJoCo course. You will describe a world in **MJCF** (MuJoCo's XML format), compile it, simulate it forward in time, inspect its state, and render a short video — all running headless on the AMD GPU through EGL.

The three objects you will use constantly:
- **`mjModel`** — the *static* description of the world, compiled from an MJCF file (bodies, joints, geoms, masses, actuators).
- **`mjData`** — the *dynamic* state that changes every step: `qpos` (positions), `qvel` (velocities), `ctrl` (controls), `xpos` (world positions), and more.
- **`mj_step`** — advance the physics by one timestep; **`mj_forward`** recomputes derived quantities without advancing time.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + JAX/MJX environment (the `auplc-mujoco-mjx` course image) that this notebook is built for.

## Goals
- Understand the roles of `mjModel`, `mjData`, `mj_step`, and `mj_forward`
- Author a minimal world in MJCF and compile it to a model
- Run a simulation loop and read back the state (`qpos`)
- Render an offscreen video on the GPU with EGL

### Import libraries

We set `MUJOCO_GL=egl` **before** importing `mujoco` so rendering happens offscreen on the GPU (no display needed). `imageio` writes the frames to an mp4.

In [ ]:
import os
# Headless offscreen rendering backend (set before importing mujoco)
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import numpy as np
import matplotlib.pyplot as plt
import mujoco
import imageio

### Describe the world in MJCF and compile it

MJCF is an XML description of a scene. Below we define a floor plus a single-hinge **pendulum** that swings under gravity. `mujoco.MjModel.from_xml_string` compiles the XML into an `mjModel`; `mujoco.MjData` allocates the matching dynamic state. We then print the model's dimensions — `nq` (number of generalized positions) and `nv` (number of generalized velocities) — which here are both 1 (the single hinge angle).

In [ ]:
# A minimal MJCF: a floor + a single-hinge pendulum that swings under gravity.
MJCF = """
<mujoco model="pendulum">
  <option gravity="0 0 -9.81" timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="2 2 0.1" rgba="0.8 0.9 0.8 1"/>
    <body name="pole" pos="0 0 1.2">
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom name="mass" type="capsule" fromto="0 0 0 0 0 -0.6" size="0.05" rgba="0.2 0.4 0.9 1"/>
    </body>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(MJCF)
data = mujoco.MjData(model)

print("nq (generalized positions):", model.nq)
print("nv (generalized velocities):", model.nv)
print("nbody:", model.nbody)
print("timestep:", model.opt.timestep)
print("joint name:", mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, 0))

### Simulate with `mj_step`

We reset the state, set the initial hinge angle to 90°, then call `mj_step` in a loop. Each call integrates the physics by one `timestep`. We record `data.time` and `data.qpos[0]` every step and plot the angle over time — you should see the classic damped/undamped pendulum oscillation.

In [ ]:
# Simulate: start the pendulum from a horizontal angle and let it swing.
mujoco.mj_resetData(model, data)
data.qpos[0] = np.pi / 2  # initial hinge angle

times, angles = [], []
for _ in range(1500):
    mujoco.mj_step(model, data)
    times.append(data.time)
    angles.append(data.qpos[0])

plt.figure(figsize=(7, 3))
plt.plot(times, angles)
plt.xlabel("time (s)")
plt.ylabel("hinge angle (rad)")
plt.title("Pendulum qpos over time")
plt.grid(True)
plt.show()

### Render the motion to a video

A `mujoco.Renderer` draws the current `mjData` into an RGB image. We re-run the swing, capture a frame every few steps (to land near 30 fps), and save them as an mp4 with `imageio`.

In [ ]:
# Render a short video of the swing (offscreen EGL).
os.makedirs("output/videos", exist_ok=True)
renderer = mujoco.Renderer(model, height=320, width=480)

mujoco.mj_resetData(model, data)
data.qpos[0] = np.pi / 2

frames = []
for i in range(600):
    mujoco.mj_step(model, data)
    if i % 4 == 0:  # ~125 fps sim -> ~30 fps video
        renderer.update_scene(data)
        frames.append(renderer.render())

video_path = "output/videos/mjx01_pendulum.mp4"
imageio.mimsave(video_path, frames, fps=30)
print(f"Saved {len(frames)} frames to {video_path}")

### Watch the result

`IPython.display.Video` embeds the saved mp4 inline in the notebook.

In [ ]:
from IPython.display import Video
Video(url="output/videos/mjx01_pendulum.mp4")

## Conclusions

You compiled a world from MJCF, stepped it through time with `mj_step`, read the state out of `mjData`, and rendered it on the GPU. These are the primitives every later notebook builds on. Next (MJX 02) we take control of cameras, geom groups, and contact visualization.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT